# 手动测试 Notebook

用于交互式测试核心功能：`chat()`、`ingest_all()`、`search()`。

**使用前确认：**
1. 已激活虚拟环境（`.venv`）
2. 已完成文档入库（`python -m rag.ingest`）
3. `.env` 文件中 API Key 配置正确


## 1. Setup & Imports

In [1]:
import os
import sys
import logging

# 将项目根目录加入 Python 路径（Notebook 从 tests/ 子目录运行时需要）
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 必须在 huggingface/transformers 相关库 import 之前加载 .env
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))
for _key in ("HF_ENDPOINT", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    _val = os.getenv(_key)
    if _val:
        os.environ[_key] = _val

# 配置日志在 notebook 内显示
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# 导入核心模块
from rag.ingest import ingest_all
from rag.retriever import search
from llm.provider import chat
import config

print(f"✅ 导入成功")
print(f"   LLM Provider : {config.ACTIVE_PROVIDER}")
print(f"   Embedding 模型: {config.EMBEDDING_MODEL}")
print(f"   ChromaDB 路径 : {config.CHROMA_DB_PATH}")
print(f"   文档目录      : {config.DOCS_DIR}")


✅ 导入成功
   LLM Provider : kimi
   Embedding 模型: all-MiniLM-L6-v2
   ChromaDB 路径 : ./chroma_db
   文档目录      : ./docs


## 2. 测试 `ingest_all()` — 文档入库

扫描 `datasets/data_en/` 目录，将所有支持格式的文档向量化并存入 ChromaDB。  
**文档有更新时重新运行此 Cell 即可，支持 upsert（不会重复入库）。**

In [ ]:
# 执行入库（日志会直接输出在 Cell 下方）
ingest_all("../datasets/data_en", model="en")


c:\DiskD\sourceCode\mygithub\AgentA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
14:00:53 [INFO] Load pretrained SentenceTransformer: all-MiniLM-L6-v2
14:00:53 [INFO] Load pretrained SentenceTransformer: all-MiniLM-L6-v2
c:\DiskD\sourceCode\mygithub\AgentA\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
14:00:54 [INFO] 发现 12 个文档，开始入库...
14:00:54 [INFO]   解析: 2026-03-11 DWA live training_Presentation.pdf
c:\DiskD\sourceCode\mygithub\AgentA\.venv\Lib\site-packages\transformers\tokenizati

In [ ]:
# 验证入库结果：查看所有 ChromaDB collection 的统计信息
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

client = chromadb.PersistentClient(path=config.CHROMA_DB_PATH)

print(f"📦 ChromaDB 存储路径: {config.CHROMA_DB_PATH}\n")
for alias, (model_name, collection_name) in config.EMBEDDING_MODELS.items():
    try:
        embedding_fn = SentenceTransformerEmbeddingFunction(model_name=model_name)
        col = client.get_collection(name=collection_name, embedding_function=embedding_fn)  # type: ignore[arg-type]
        total: int = col.count()
        print(f"  [{alias}] {collection_name:<10}  模型: {model_name:<30}  共 {total} 块")

        # 展示前 3 条样本
        sample = col.get(limit=3, include=["documents", "metadatas"])
        docs = sample["documents"] or []
        metas = sample["metadatas"] or []
        for i, (doc, meta) in enumerate(zip(docs, metas), start=1):
            source = meta.get("source", "unknown")
            chunk_idx = meta.get("chunk_index", "?")
            preview = str(doc)[:80].replace("\n", " ")
            print(f"    [{i}] 来源: {source}  块: {chunk_idx}  内容: {preview}...")
        print()
    except Exception:
        print(f"  [{alias}] {collection_name} — 尚未入库\n")


📦 ChromaDB collection 'knowledge_base' 当前共 79 个文本块

--- 前 5 条样本 ---
[1] 来源: 2026-03-11 DWA live training_Presentation.pdf  块序号: 0
    内容预览: Nokia internal use © 2026 Nokia2 Learn to #worksmart with Digital Workplace Adop...

[2] 来源: 2026-03-11 DWA live training_Presentation.pdf  块序号: 1
    内容预览: 2026 Nokia4 Learn how to AI with DWA Learn to #worksmart with Digital Workplace ...

[3] 来源: 2026-03-11 DWA live training_Presentation.pdf  块序号: 2
    内容预览: ousekeeping Recording Training is being recorded &  uploaded to DWA live  traini...

[4] 来源: 2026-03-11 DWA live training_Presentation.pdf  块序号: 3
    内容预览: Today’s topic: Modular Prompting: a simpler way to collaborate with AI Agenda: 1...

[5] 来源: 2026-03-11 DWA live training_Presentation.pdf  块序号: 4
    内容预览: Join us on Viva Engage! Our objective today • Improve your confidence and capabi...



## 3. 测试 `search()` — 向量检索

输入自然语言问题，返回 ChromaDB 中最相关的文档片段（含来源文件名和相似度）。  
可修改 `top_k` 和 `queries` 列表来观察不同检索结果。

In [5]:
# ✏️ 修改这里来测试不同的查询
queries: list[str] = [
    "master 是谁",
    "ChromaDB 有什么特点",
    "支持哪些 LLM 模型",
]
top_k: int = 3  # 每次返回最相关的 top_k 个片段

for query in queries:
    print(f"{'='*60}")
    print(f"🔍 查询: {query}")
    print(f"{'='*60}")
    result = search(query, top_k=top_k)
    print(result)
    print()


🔍 查询: master 是谁


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.76it/s]



[1] 来源: readme.txt（相似度: 0.2874）
I'm the master of this project, my name is Michael.

---

[2] 来源: test_sample.md（相似度: 0.2803）
# 私有知识库测试文档

## 项目简介

本项目是一个基于 RAG（检索增强生成）技术的私有知识库 Agent。

## 核心功能

- 支持多格式文档解析（MD、PDF、Word、Excel 等）
- 本地向量化存储，数据不出本地
- 自然语言提问，自动检索相关片段
- 可切换 LLM 提供商

## 技术栈

- Python 3.11
- ChromaDB 向量数据库
- sentence-transformers 嵌入模型
- OpenAI / Kimi / DeepSeek LLM

---

[3] 来源: test_sample.docx（相似度: 0.2415）
LLM 大语言模型简介

大语言模型（LLM）是基于 Transformer 架构的深度学习模型。

代表模型：GPT-4、Claude、Gemini、Kimi、DeepSeek。

🔍 查询: ChromaDB 有什么特点


Batches: 100%|██████████| 1/1 [00:00<00:00, 72.08it/s]



[1] 来源: test_sample.html（相似度: 0.6585）
向量数据库介绍

向量数据库是专门用于存储和检索高维向量数据的数据库系统。

主要特点

支持高效的相似度搜索（ANN 算法）

适合存储文本、图像、音频的嵌入向量

常见产品：ChromaDB、Pinecone、Weaviate、Milvus

ChromaDB

ChromaDB 是一款开源的本地向量数据库，零配置，Python 原生支持，适合个人和小团队使用。

---

[2] 来源: test_sample.md（相似度: 0.5261）
# 私有知识库测试文档

## 项目简介

本项目是一个基于 RAG（检索增强生成）技术的私有知识库 Agent。

## 核心功能

- 支持多格式文档解析（MD、PDF、Word、Excel 等）
- 本地向量化存储，数据不出本地
- 自然语言提问，自动检索相关片段
- 可切换 LLM 提供商

## 技术栈

- Python 3.11
- ChromaDB 向量数据库
- sentence-transformers 嵌入模型
- OpenAI / Kimi / DeepSeek LLM

---

[3] 来源: test_sample.txt（相似度: 0.4062）
RAG 技术简介

RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合信息检索与大语言模型的技术。

工作流程：
1. 将私有文档向量化，存入向量数据库
2. 用户提问时，将问题向量化并检索相似文档片段
3. 将检索到的文档片段作为上下文传给 LLM
4. LLM 基于上下文生成准确答案

优势：
- 减少 LLM 幻觉（Hallucination）
- 知识可实时更新，无需重新训练模型
- 数据私有，不上传到云端

🔍 查询: 支持哪些 LLM 模型


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.35it/s]

[1] 来源: test_sample.pptx（相似度: 0.4775）
[Slide 1]
Agent 技术概述
Agent 通过 ReAct 循环，结合工具调用完成复杂任务。

[Slide 2]
Function Calling
LLM 通过 Function Calling 调用外部工具，如搜索、计算、数据库查询。

---

[2] 来源: test_sample.md（相似度: 0.3857）
# 私有知识库测试文档

## 项目简介

本项目是一个基于 RAG（检索增强生成）技术的私有知识库 Agent。

## 核心功能

- 支持多格式文档解析（MD、PDF、Word、Excel 等）
- 本地向量化存储，数据不出本地
- 自然语言提问，自动检索相关片段
- 可切换 LLM 提供商

## 技术栈

- Python 3.11
- ChromaDB 向量数据库
- sentence-transformers 嵌入模型
- OpenAI / Kimi / DeepSeek LLM

---

[3] 来源: test_sample.docx（相似度: 0.3528）
LLM 大语言模型简介

大语言模型（LLM）是基于 Transformer 架构的深度学习模型。

代表模型：GPT-4、Claude、Gemini、Kimi、DeepSeek。



## 4. 测试 `chat()` — LLM 对话

直接调用 LLM，不经过知识库检索。用于验证 API 连通性和模型响应质量。

In [6]:
# ✏️ 修改这里来测试不同的问题
questions: list[str] = [
    "用一句话介绍你自己",
    "什么是向量数据库？",
]

for question in questions:
    print(f"{'='*60}")
    print(f"💬 问题: {question}")
    print(f"{'='*60}")
    response = chat([{"role": "user", "content": question}])
    print(f"🤖 回答: {response}")
    print()


💬 问题: 用一句话介绍你自己


14:12:06 [INFO] HTTP Request: POST https://api.moonshot.cn/v1/chat/completions "HTTP/1.1 200 OK"


🤖 回答: 你好，我是一个由人工智能驱动的助手，擅长中英文对话，致力于提供安全、有帮助、准确的回答。

💬 问题: 什么是向量数据库？


14:12:13 [INFO] HTTP Request: POST https://api.moonshot.cn/v1/chat/completions "HTTP/1.1 200 OK"


🤖 回答: 向量数据库是一种专门用于存储、管理、检索和操作向量数据的数据库系统。与关系型数据库不同，向量数据库主要关注于非结构化数据，如图像、视频、音频等。这些数据通常以向量形式表示，用于描述数据的特征和属性。向量数据库的关键在于高效地处理和检索这些向量数据，以便为用户提供快速准确的查询结果。

向量数据库的主要特点如下：

1. 向量表示：向量数据库中的数据以向量形式存储，这些向量可以是高维的，表示复杂的数据结构和特征。常见的向量表示方法包括词向量、图像向量、音频向量等。

2. 相似度搜索：向量数据库的主要任务之一是进行相似度搜索，即在数据库中找到与给定向量最相似的数据。这通常涉及到余弦相似度、欧氏距离等计算方法。

3. 索引技术：为了提高检索效率，向量数据库采用特定的索引技术，如KD树、球树、哈希表等。这些索引技术可以加速相似度搜索，降低查询时间。

4. 可扩展性：向量数据库需要能够处理大规模数据集，因此需要具备良好的可扩展性。这包括水平扩展（增加更多的服务器节点）和垂直扩展（增强单个服务器的性能）。

5. 分布式处理：为了提高性能和可靠性，向量数据库通常采用分布式架构。这意味着数据和计算任务可以在多个服务器之间分布，以提高系统的吞吐量和容错能力。

总之，向量数据库是一种专门用于处理和检索向量数据的数据库系统，它在许多领域（如机器学习、计算机视觉、自然语言处理等）具有广泛的应用前景。



In [ ]:
# 测试带 system prompt 的对话
messages: list[dict] = [
    {"role": "system", "content": "你是一个私有知识库助手，请简洁、准确地回答问题。"},
    {"role": "user", "content": "RAG 和直接调用 LLM 的区别是什么？"},
]
response = chat(messages)
print(f"🤖 带 System Prompt 的回答:\n{response}")


## 5. 对比测试：`search()` vs `chat()` 并排比较

将同一批问题同时送入 `search()` 和 `chat()`，结果用 pandas DataFrame 并排展示，  
便于直观比较"知识库检索到的原文"与"LLM 生成的回答"之间的差异。

In [ ]:
import pandas as pd

# ✏️ 修改这里来批量对比测试
batch_queries: list[str] = [
    "RAG 技术是什么",
    "ChromaDB 有什么特点",
    "支持哪些 LLM 模型",
    "向量数据库如何工作",
]

rows: list[dict] = []
for query in batch_queries:
    print(f"处理中: {query} ...")
    search_result: str = search(query, top_k=2)
    chat_response: str = chat([{"role": "user", "content": query}])
    rows.append({
        "查询问题": query,
        "search() 检索结果（来自知识库）": search_result[:200] + "..." if len(search_result) > 200 else search_result,
        "chat() 直接回答（LLM 生成）": chat_response,
    })

df = pd.DataFrame(rows)

# 设置显示选项，避免内容被截断
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
df
